In [2]:
import tkinter as tk
from tkinter import ttk, messagebox

In [3]:
# ──────────────────────────────────────────────
#  DOMAIN / BUSINESS LOGIC LAYER
# ──────────────────────────────────────────────

class DeliveryLocation:
    """Represents a delivery destination with its pricing rules."""

    def __init__(self, name: str, heavy_price: float, light_price: float,
                 weight_threshold: float = 10.0):
        self.name = name
        self.heavy_price = heavy_price          # price when weight >= threshold
        self.light_price = light_price          # price when weight < threshold
        self.weight_threshold = weight_threshold

    def calculate_charge(self, weight: float) -> float:
        """Return the delivery charge for a given package weight."""
        if weight >= self.weight_threshold:
            return self.heavy_price
        return self.light_price

    def __str__(self):
        return self.name


class DeliveryService:
    """Manages a collection of delivery locations and pricing."""

    def __init__(self):
        self._locations: dict[str, DeliveryLocation] = {}

    def add_location(self, location: DeliveryLocation) -> None:
        self._locations[location.name] = location

    def get_location_names(self) -> list[str]:
        return list(self._locations.keys())

    def get_charge(self, location_name: str, weight: float) -> float:
        if location_name not in self._locations:
            raise ValueError(f"Unknown location: {location_name}")
        return self._locations[location_name].calculate_charge(weight)

    def get_location(self, name: str) -> DeliveryLocation:
        return self._locations[name]


# ──────────────────────────────────────────────
#  GUI LAYER
# ──────────────────────────────────────────────

class DeliveryApp(tk.Tk):
    """Main application window — owns the service and all frames."""

    # ── brand colours ──────────────────────────
    BG          = "#0D1117"
    SURFACE     = "#161B22"
    ACCENT      = "#F78166"
    ACCENT2     = "#58A6FF"
    TEXT        = "#E6EDF3"
    MUTED       = "#8B949E"
    SUCCESS     = "#3FB950"
    BORDER      = "#30363D"

    def __init__(self):
        super().__init__()
        self.title("SwiftDrop — Delivery Calculator")
        self.resizable(False, False)
        self.configure(bg=self.BG)

        # centre window
        w, h = 480, 560
        sw, sh = self.winfo_screenwidth(), self.winfo_screenheight()
        self.geometry(f"{w}x{h}+{(sw-w)//2}+{(sh-h)//2}")

        # build service
        self.service = self._build_service()

        # build UI
        self._build_ui()

    # ── service factory ────────────────────────
    @staticmethod
    def _build_service() -> DeliveryService:
        svc = DeliveryService()
        svc.add_location(DeliveryLocation("PAU",  heavy_price=2000, light_price=1500))
        svc.add_location(DeliveryLocation("Epe",  heavy_price=5000, light_price=4000))
        return svc

    # ── UI construction ────────────────────────
    def _build_ui(self):
        # ── header ────────────────────────────
        hdr = tk.Frame(self, bg=self.SURFACE, pady=20)
        hdr.pack(fill="x")

        tk.Label(hdr, text="📦  SwiftDrop", font=("Courier New", 22, "bold"),
                 bg=self.SURFACE, fg=self.ACCENT).pack()
        tk.Label(hdr, text="Delivery Cost Estimator",
                 font=("Courier New", 10), bg=self.SURFACE, fg=self.MUTED).pack()

        # ── divider ───────────────────────────
        tk.Frame(self, bg=self.BORDER, height=1).pack(fill="x")

        # ── form card ─────────────────────────
        card = tk.Frame(self, bg=self.BG, padx=40, pady=30)
        card.pack(fill="both", expand=True)

        self._label(card, "DELIVERY LOCATION").pack(anchor="w")
        self.location_var = tk.StringVar(value=self.service.get_location_names()[0])
        self._location_menu(card).pack(fill="x", pady=(6, 20))

        self._label(card, "PACKAGE WEIGHT (kg)").pack(anchor="w")
        self.weight_var = tk.StringVar()
        self._weight_entry(card).pack(fill="x", pady=(6, 8))

        # threshold hint
        self.hint_var = tk.StringVar()
        tk.Label(card, textvariable=self.hint_var, font=("Courier New", 9),
                 bg=self.BG, fg=self.MUTED).pack(anchor="w")

        self._update_hint()  # initial hint

        # ── calculate button ──────────────────
        tk.Frame(card, bg=self.BG, height=20).pack()
        calc_btn = tk.Button(
            card, text="CALCULATE CHARGE  →",
            font=("Courier New", 11, "bold"),
            bg=self.ACCENT, fg=self.BG, activebackground="#d9604a",
            relief="flat", cursor="hand2", pady=12,
            command=self._calculate
        )
        calc_btn.pack(fill="x")

        # ── result panel ──────────────────────
        tk.Frame(card, bg=self.BG, height=20).pack()
        self.result_frame = tk.Frame(card, bg=self.SURFACE,
                                     highlightbackground=self.BORDER,
                                     highlightthickness=1, pady=20)
        self.result_frame.pack(fill="x")

        tk.Label(self.result_frame, text="AMOUNT DUE",
                 font=("Courier New", 9, "bold"),
                 bg=self.SURFACE, fg=self.MUTED).pack()

        self.result_var = tk.StringVar(value="—")
        tk.Label(self.result_frame, textvariable=self.result_var,
                 font=("Courier New", 30, "bold"),
                 bg=self.SURFACE, fg=self.SUCCESS).pack()

        self.breakdown_var = tk.StringVar()
        tk.Label(self.result_frame, textvariable=self.breakdown_var,
                 font=("Courier New", 9), bg=self.SURFACE, fg=self.MUTED).pack()

        # ── footer ────────────────────────────
        tk.Label(self, text="Prices include VAT • Rates updated 2025",
                 font=("Courier New", 8), bg=self.BG, fg=self.BORDER).pack(pady=10)

        # trace location change for live hint
        self.location_var.trace_add("write", lambda *_: self._update_hint())

    # ── widget helpers ─────────────────────────
    def _label(self, parent, text):
        return tk.Label(parent, text=text, font=("Courier New", 9, "bold"),
                        bg=self.BG, fg=self.MUTED)

    def _location_menu(self, parent):
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("Custom.TCombobox",
                        fieldbackground=self.SURFACE,
                        background=self.SURFACE,
                        foreground=self.TEXT,
                        selectbackground=self.SURFACE,
                        selectforeground=self.ACCENT2,
                        bordercolor=self.BORDER,
                        arrowcolor=self.ACCENT2)
        cb = ttk.Combobox(parent, textvariable=self.location_var,
                          values=self.service.get_location_names(),
                          state="readonly", style="Custom.TCombobox",
                          font=("Courier New", 13))
        return cb

    def _weight_entry(self, parent):
        frame = tk.Frame(parent, bg=self.SURFACE,
                         highlightbackground=self.BORDER,
                         highlightthickness=1)
        tk.Entry(frame, textvariable=self.weight_var,
                 font=("Courier New", 16), bg=self.SURFACE,
                 fg=self.TEXT, insertbackground=self.ACCENT2,
                 relief="flat", bd=10).pack(side="left", fill="x", expand=True)
        tk.Label(frame, text="kg", font=("Courier New", 12),
                 bg=self.SURFACE, fg=self.MUTED, padx=10).pack(side="right")
        return frame

    # ── logic ──────────────────────────────────
    def _update_hint(self):
        loc_name = self.location_var.get()
        loc = self.service.get_location(loc_name)
        self.hint_var.set(
            f"Below {loc.weight_threshold:.0f} kg → ₦{loc.light_price:,.0f}  │  "
            f"{loc.weight_threshold:.0f} kg & above → ₦{loc.heavy_price:,.0f}"
        )

    def _calculate(self):
        loc_name = self.location_var.get()
        raw = self.weight_var.get().strip()

        if not raw:
            messagebox.showwarning("Missing Input", "Please enter the package weight.")
            return

        try:
            weight = float(raw)
            if weight <= 0:
                raise ValueError
        except ValueError:
            messagebox.showerror("Invalid Weight",
                                 "Please enter a valid positive number for the weight.")
            return

        charge = self.service.get_charge(loc_name, weight)
        loc = self.service.get_location(loc_name)
        tier = "heavy" if weight >= loc.weight_threshold else "light"

        self.result_var.set(f"₦{charge:,.0f}")
        self.breakdown_var.set(
            f"Location: {loc_name}  •  Weight: {weight:.2f} kg  •  Tier: {tier}"
        )


# ──────────────────────────────────────────────
#  ENTRY POINT
# ──────────────────────────────────────────────

if __name__ == "__main__":
    app = DeliveryApp()
    app.mainloop()